# Trashpanda: models that carry their provenance
Choose tiny **Llama** or **GPT-2**, prepare synthetic data, train across sessions, and export native Hugging Face weights with inspectable provenance.

The dropdown uses the same architecture adapters as `trashpanda train --model llama` or `--model gpt2`; Python APIs expose the checkpoint interval and session controls directly.

These are randomly initialized demonstration models with a 32-token vocabulary, not pretrained chat models. CPU is sufficient; a GPU runtime is optional. Installation requires internet, but training and provenance verification use local files.

**After a timeout:** reconnect, run from the top, authorize Drive again, and use the **same run name and settings**. Training resumes from the latest readable checkpoint. Repeat the training cell to advance another chunk. The total token budget is fixed when creating the run; use a new name to change it or the architecture. The small default is a quick walkthrough; larger budgets can span sessions.

Only run one notebook/runtime against a given run folder at a time. Checkpoints include optimizer, scheduler, sampler and RNG state. Work since the last saved checkpoint may be lost on an abrupt disconnect. Drive mounting is not a guarantee that buffered writes reached remote storage; keep earlier checkpoints. See [Colab's Drive and runtime limits](https://research.google.com/colaboratory/faq.html).

In [ ]:
#@title 1. Connect your Google Drive (authorize the prompt)
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/trashpanda")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Connect Google Drive before continuing.")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage:", DRIVE_ROOT)

In [ ]:
#@title 2. Select a model and persistent run
model_architecture = "llama" #@param ["llama", "gpt2"]
run_name = "my-first-run" #@param {type:"string"}
token_budget = 12400 #@param {type:"integer"}
steps_per_session = 25 #@param {type:"integer"}
checkpoint_every = 5 #@param {type:"integer"}

import json
import os
import re
import uuid

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", run_name):
    raise ValueError("Use a run name containing only letters, numbers, underscores or hyphens.")
if model_architecture not in {"llama", "gpt2"}:
    raise ValueError("Select llama or gpt2.")
if any(type(v) is not int or v <= 0 for v in (token_budget, steps_per_session, checkpoint_every)):
    raise ValueError("Budget, session steps, and checkpoint interval must be positive integers.")
RUN = DRIVE_ROOT / run_name
RUN.mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)

# Pin the demonstrated implementation across runtime restarts.
CODE_REVISION = "acc418fc9e41e42d885dca1dc8baf9aa075d4290"
settings = {"architecture": model_architecture, "token_budget": token_budget,
            "checkpoint_interval": checkpoint_every, "code_revision": CODE_REVISION}
settings_path = RUN / "settings.json"
if settings_path.exists():
    if json.loads(settings_path.read_text()) != settings:
        raise ValueError("This run has different saved settings. Restore them or choose a new run name. "
                         + settings_path.read_text())
else:
    write_json(settings_path, settings)
print("Run folder:", RUN)
print("Settings:", settings)

In [ ]:
#@title 3. Install the pinned source revision
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Colab runtime with Python 3.11 or newer.")
REPO = Path("/content/trashpanda-source")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/rawkintrevo/trashpanda.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "origin", settings["code_revision"]], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", settings["code_revision"]], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# Record each runtime's dependencies rather than overwriting earlier sessions.
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN / ("environment-" + uuid.uuid4().hex + ".txt")).write_text(versions)
print("Installed revision:", settings["code_revision"])

## Persist inputs and preparation
Source JSONL, prepared token shards, tokenizer snapshots, configuration, checkpoints, metrics, and exports all live below the Drive run folder. A completed preparation is reused. Interrupted preparation/export attempts remain in uniquely named `.partial-*` folders; they are never treated as completed artifacts.

The synthetic source IDs and byte hashes become input records. Preparation records describe tokenization and source attribution; training adds run/session records. Nothing here asserts legal approval or trusted authorship.

In [ ]:
#@title 4. Prepare persistent synthetic inputs and configuration
from dataclasses import asdict, replace
import torch
from trashpanda.config import model_recipe, config_from_dict
from trashpanda.prepare import prepare_splits
from trashpanda.tokenizer import TinyWordLevelTokenizer
from trashpanda.data import SequenceReader

# Limit CPU overhead for these very small models.
torch.set_num_threads(2)
SOURCES = RUN / "sources"
SOURCES.mkdir(exist_ok=True)
for split, documents, words, offset in (("train", 8, 128, 0), ("validation", 4, 96, 13)):
    path = SOURCES / (split + ".jsonl")
    expected = "".join(json.dumps({"text": " ".join(f"w{(i + d + offset) % 29}" for i in range(words))}) + "\n"
                       for d in range(documents))
    if path.exists() and path.read_text() != expected:
        raise ValueError(f"Source changed or incomplete: {path}. Use a new run name or restore the file.")
    if not path.exists():
        temporary = path.with_suffix(".partial-" + uuid.uuid4().hex)
        temporary.write_text(expected)
        temporary.replace(path)

PREPARED = RUN / "prepared"
if not PREPARED.exists():
    staging = RUN / ("prepared.partial-" + uuid.uuid4().hex)
    prepare_splits(staging, train_path=SOURCES / "train.jsonl",
                   validation_path=SOURCES / "validation.jsonl",
                   tokenizer=TinyWordLevelTokenizer(), context_length=32,
                   source_descriptors=[{"source_id": "synthetic:" + split,
                                        "source_revision": "fixture-1", "evidence_state": "observed"}
                                       for split in ("train", "validation")])
    staging.rename(PREPARED)

base = model_recipe("tiny", model_architecture)
requested_config = replace(base,
    training=replace(base.training, token_budget=token_budget,
                     checkpoint_interval=checkpoint_every, device="auto"),
    runtime=replace(base.runtime, output_dir=str(RUN / "checkpoints")))
config_path = RUN / "config.json"
if config_path.exists():
    config = config_from_dict(json.loads(config_path.read_text()))
    if config.to_dict() != requested_config.to_dict():
        raise ValueError("Saved configuration differs; use a new run name.")
else:
    config = requested_config
    write_json(config_path, config.to_dict())
dataset = SequenceReader(PREPARED, split="train")
print(json.dumps(config.to_dict(), indent=2))
print("Prepared manifest:", PREPARED / "manifest.json")

## Train a chunk, then resume
`steps_per_session` is the number of **additional updates** for this invocation. You can change it between sessions. Checkpoints are written directly to Drive every `checkpoint_every` updates and at the end of a chunk; the full token budget need not finish in one runtime.

Run the next cell again to continue, or reconnect and rerun from the top after a timeout. The code scans checkpoints newest first, excludes temporary files, and reports unreadable checkpoints before trying an older one. If all checkpoints are unreadable it stops instead of silently starting over. Existing checkpoints are retained. A changed runtime/dependency stack can affect numerical reproducibility; every session saves its package versions.

In [ ]:
#@title 5. Train or resume the next chunk
from trashpanda.checkpoint import list_checkpoints, load_checkpoint
from trashpanda.model import build_model
from trashpanda.train import train

CHECKPOINTS = RUN / "checkpoints"

def latest_usable_checkpoint():
    candidates = list_checkpoints(CHECKPOINTS)
    probe = build_model(config.model, seed=config.training.seed) if candidates else None
    for path in reversed(candidates):
        try:
            payload = load_checkpoint(path, model=probe)
            # Ensure this has the state required for a training restart.
            for key in ("state", "optimizer", "sampler", "rng", "config"):
                if key not in payload:
                    raise ValueError(f"Missing checkpoint field: {key}")
        except Exception as exc:
            print(f"Skipping unreadable checkpoint {path.name}: {exc}")
            continue
        if payload["config"]["training"] != config.to_dict()["training"]:
            raise ValueError("Checkpoint training settings differ from this run.")
        if payload["config"]["model"] != config.to_dict()["model"]:
            raise ValueError("Checkpoint architecture differs from this run.")
        if payload.get("data_identity") != dataset.manifest.get("identity"):
            raise ValueError("Checkpoint data identity differs from prepared data.")
        return path, payload["state"]
    if candidates:
        raise RuntimeError("No usable checkpoint. Restore a backup or choose a new run name.")
    return None, {"update_step": 0, "consumed_tokens": 0}

checkpoint, state = latest_usable_checkpoint()
if state["consumed_tokens"] >= config.training.token_budget:
    print("Budget already completed; no additional training needed.")
else:
    print("Resuming:" if checkpoint else "Starting:", checkpoint or model_architecture)
    result = train(config, dataset, tokenizer=TinyWordLevelTokenizer(),
                   resume=checkpoint, checkpoint_dir=CHECKPOINTS,
                   max_steps=state["update_step"] + steps_per_session)
    checkpoint, state = result.checkpoint, result.state.to_dict()
    del result
print("Checkpoint:", checkpoint)
print("Progress:", state)
print("Metrics:", CHECKPOINTS / "metrics.jsonl")
print("Rerun this cell for another chunk, or continue below to export current progress.")

In [ ]:
#@title 6. Evaluate, sample, and export the latest checkpoint with provenance
from trashpanda.evaluate import evaluate, sample_text
from trashpanda.export import export_checkpoint
from trashpanda.provenance import inspect_bundle, verify_bundle, sha256_file

checkpoint, state = latest_usable_checkpoint()
if checkpoint is None:
    raise RuntimeError("Run a training chunk before exporting.")
model = build_model(config.model)
load_checkpoint(checkpoint, model=model)
validation = SequenceReader(PREPARED, split="validation")
metrics = evaluate(model, validation, max_batches=2)
print("Evaluation:", metrics)
print("Sample:", sample_text(model, TinyWordLevelTokenizer(), "w0 w1", max_new_tokens=8, seed=42))
write_json(RUN / f"evaluation-{state['update_step']:08d}.json", asdict(metrics))

# Include the checkpoint digest so recovered/replayed steps cannot collide.
EXPORTS = RUN / "exports"
EXPORTS.mkdir(exist_ok=True)
EXPORT = EXPORTS / (checkpoint.stem + "-" + sha256_file(checkpoint)[:12])
if not EXPORT.exists():
    staging = EXPORTS / (EXPORT.name + ".partial-" + uuid.uuid4().hex)
    export_checkpoint(checkpoint, staging)
    verification = verify_bundle(staging)
    if not verification.ok:
        raise RuntimeError(verification.to_dict())
    staging.rename(EXPORT)
print("Portable Hugging Face export:", EXPORT)
print(json.dumps(inspect_bundle(EXPORT), indent=2))
print(json.dumps(verify_bundle(EXPORT).to_dict(), indent=2))

The export contains model/tokenizer files, `provenance-manifest.json`, and digest-addressed records in `provenance/records/`. It carries input, preparation and run evidence from the checkpoint. The evaluation above is saved separately; it is not retroactively added to checkpoint evidence.

Verification distinguishes integrity, evidence completeness, signatures, and policy. **Unsigned means unauthenticated**, even when hashes verify. No policy means authorization has not been evaluated. An intermediate export represents only the updates completed so far, not the requested full budget.

In [ ]:
#@title 7. Verify a relocated copy and demonstrate tamper detection
import shutil
import tempfile
from transformers import AutoModelForCausalLM, AutoTokenizer

# Scratch copies only: never modify the original Drive export.
with tempfile.TemporaryDirectory() as temporary:
    relocated = Path(temporary) / "export"
    shutil.copytree(EXPORT, relocated)
    assert verify_bundle(relocated).ok
    restored = AutoModelForCausalLM.from_pretrained(relocated, local_files_only=True, trust_remote_code=False)
    tokenizer = AutoTokenizer.from_pretrained(relocated, local_files_only=True, trust_remote_code=False)
    print("Offline reload:", restored.config.model_type, "vocabulary:", tokenizer.vocab_size)
    payload = relocated / "config.json"
    with payload.open("a") as handle:
        handle.write("\n ")
    tampered = verify_bundle(relocated)
    assert not tampered.ok
    print("Tampered copy integrity:", tampered.integrity)
print("Original Drive bundle remains valid:", verify_bundle(EXPORT).ok)

## Returning after a runtime expires
1. Open this notebook, reconnect, and run cells 1–5 with the same run name and settings. Drive will prompt for authorization when needed.
2. Check the printed resumed checkpoint and counters. Repeat cell 5 to train more; run cells 6–7 whenever you want an export.
3. Find all durable artifacts under `MyDrive/trashpanda/<run_name>/`. Download the **whole export folder**, including `provenance/`, to carry the evidence with the model.

Keep earlier checkpoints until you have confirmed a later one is usable. Interrupted `.partial-*` artifacts can be removed manually once no runtime is using the run. Drive I/O/quota errors must be resolved before continuing. Metrics may contain replayed steps after recovery; checkpoint counters are the authority for committed progress. Never run two writers against the same folder.

This walkthrough does not automatically select the large A100 recipe: GPT-2 currently supports only `tiny`, and the production recipe has different data, tokenizer, and hardware requirements.